In [1]:
from pathlib import Path
import pandas as pd

# 1. Đọc dữ liệu đã đánh giá của 50 run
PROJECT_DIR = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "results/evaluation/aoea_summary.csv").is_file()),
    None
)
assert PROJECT_DIR is not None, "Không tìm thấy thư mục dự án"

EVAL_DIR = PROJECT_DIR / "results/evaluation"
OUT_DIR = PROJECT_DIR / "results/extended_analysis/paper_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ea = pd.read_csv(EVAL_DIR / "execution_accuracy_all_runs.csv")
assert len(ea) == 4300 and ea["Run"].nunique() == 50

overall = pd.read_csv(EVAL_DIR / "aoea_summary.csv")
by_quadrant = pd.read_csv(EVAL_DIR / "aoea_by_quadrant.csv")
assert len(overall) == 2 and len(by_quadrant) == 8
assert set(overall["Method"]) == {"SQL", "SPARQL"}

# 2. Kết quả GPT-4 trong notebook gốc, đơn vị phần trăm
# Giữ nguyên số liệu tác giả đã nhập; không tự thay đổi nguồn đối chiếu.
paper = pd.DataFrame([
    ("Overall", 16.7, 54.2),
    ("LQLS", 25.5, 71.1),
    ("HQLS", 37.4, 66.9),
    ("LQHS", 0.0, 35.7),
    ("HQHS", 0.0, 38.7)
], columns=["Quadrant", "SQL", "SPARQL"])
paper = paper.melt(id_vars="Quadrant", var_name="Method", value_name="GPT-4 (%)")

# 3. Đọc kết quả GPT-5.4 trực tiếp từ Evaluation 50 run
overall["Quadrant"] = "Overall"
ours = pd.concat([overall, by_quadrant], ignore_index=True)
assert len(ours) == 10 and not ours.duplicated(["Quadrant", "Method"]).any()
assert ours["AOEA"].between(0, 1).all()
ours["GPT-5.4 (%)"] = ours["AOEA"] * 100

comparison = paper.merge(
    ours[["Quadrant", "Method", "GPT-5.4 (%)"]],
    on=["Quadrant", "Method"],
    how="left",
    validate="one_to_one"
)
assert len(comparison) == 10 and comparison["GPT-5.4 (%)"].notna().all()
comparison["Difference (pp)"] = comparison["GPT-5.4 (%)"] - comparison["GPT-4 (%)"]

# 4. Lưu bảng so sánh; làm tròn chỉ khi xuất
comparison = comparison.round({"GPT-4 (%)": 2, "GPT-5.4 (%)": 2, "Difference (pp)": 2})
comparison.to_csv(OUT_DIR / "paper_comparison.csv", index=False, encoding="utf-8-sig")

print("=== GPT-4 (PAPER) VS GPT-5.4 (50 RUNS) ===")
display(comparison)
print("Saved: paper_comparison.csv")

=== GPT-4 (PAPER) VS GPT-5.4 (50 RUNS) ===


,Quadrant,Method,GPT-4 (%),GPT-5.4 (%),Difference (pp)
0,Overall,SQL,16.7,45.72,29.02
1,LQLS,SQL,25.5,66.00,40.50
2,HQLS,SQL,37.4,52.73,15.33
3,LQHS,SQL,0.0,9.80,9.80
4,HQHS,SQL,0.0,49.60,49.60
5,Overall,SPARQL,54.2,65.53,11.33
6,LQLS,SPARQL,71.1,80.67,9.57
7,HQLS,SPARQL,66.9,66.91,0.01
8,LQHS,SPARQL,35.7,67.20,31.50
9,HQHS,SPARQL,38.7,44.20,5.50


Saved: paper_comparison.csv


In [2]:
# 1. Đối chiếu chênh lệch tổng thể SPARQL - SQL
results = comparison.loc[comparison["Quadrant"] == "Overall"].set_index("Method")
assert set(results.index) == {"SQL", "SPARQL"}

paper_gap = float(results.loc["SPARQL", "GPT-4 (%)"] - results.loc["SQL", "GPT-4 (%)"])
our_gap = float(results.loc["SPARQL", "GPT-5.4 (%)"] - results.loc["SQL", "GPT-5.4 (%)"])
sql_gain = float(results.loc["SQL", "Difference (pp)"])
sparql_gain = float(results.loc["SPARQL", "Difference (pp)"])

gap_summary = pd.DataFrame({
    "Metric": [
        "SQL improvement",
        "SPARQL improvement",
        "SPARQL - SQL gap (GPT-4)",
        "SPARQL - SQL gap (GPT-5.4)",
        "Change in gap"
    ],
    "Value (pp)": [
        sql_gain, sparql_gain, paper_gap, our_gap, our_gap - paper_gap
    ]
}).round(2)

print("=== OVERALL COMPARISON (pp) ===")
display(gap_summary)
gap_summary.to_csv(OUT_DIR / "kg_advantage_comparison.csv", index=False, encoding="utf-8-sig")
print("Chênh lệch SPARQL - SQL là so sánh quan sát được, không chứng minh quan hệ nhân quả.")

=== OVERALL COMPARISON (pp) ===


,Metric,Value (pp)
0,SQL improvement,29.02
1,SPARQL improvement,11.33
2,SPARQL - SQL gap (GPT-4),37.50
3,SPARQL - SQL gap (GPT-5.4),19.81
4,Change in gap,-17.69


Chênh lệch SPARQL - SQL là so sánh quan sát được, không chứng minh quan hệ nhân quả.


In [3]:
# 1. So sánh chênh lệch SPARQL - SQL trong từng Quadrant
quadrants = ["LQLS", "HQLS", "LQHS", "HQHS"]
subset = comparison.loc[comparison["Quadrant"].isin(quadrants)]

paper_table = subset.pivot(index="Quadrant", columns="Method", values="GPT-4 (%)")
our_table = subset.pivot(index="Quadrant", columns="Method", values="GPT-5.4 (%)")

result = pd.DataFrame({
    "GPT-4 KG Advantage": paper_table["SPARQL"] - paper_table["SQL"],
    "GPT-5.4 KG Advantage": our_table["SPARQL"] - our_table["SQL"]
}).loc[quadrants]
result["Change (pp)"] = result["GPT-5.4 KG Advantage"] - result["GPT-4 KG Advantage"]
result = result.round(2)
assert len(result) == 4 and result.notna().all().all()

result.to_csv(OUT_DIR / "kg_advantage_by_quadrant.csv", encoding="utf-8-sig")
print("=== SPARQL - SQL GAP BY QUADRANT (pp) ===")
display(result)
print("Largest decrease:", result["Change (pp)"].idxmin())
print("Largest increase:", result["Change (pp)"].idxmax())

=== SPARQL - SQL GAP BY QUADRANT (pp) ===


,GPT-4 KG Advantage,GPT-5.4 KG Advantage,Change (pp)
Quadrant,,,
LQLS,45.6,14.67,-30.93
HQLS,29.5,14.18,-15.32
LQHS,35.7,57.40,21.70
HQHS,38.7,-5.40,-44.10


Largest decrease: HQHS
Largest increase: LQHS


In [4]:
# 1. Đọc Error Analysis đã được tính lại cho 50 run
error_file = (
    PROJECT_DIR / "results/extended_analysis/error_analysis/error_details.csv"
)
assert error_file.is_file(), f"Chưa chạy 01_error_analysis: {error_file}"
errors = pd.read_csv(error_file)
assert len(errors) == 4300 and errors["Run"].nunique() == 50

# 2. Phân loại kết quả theo Quadrant và phương pháp
outcome_order = ["Correct", "Wrong Result", "Execution Failure"]
summary = pd.crosstab(
    [errors["Quadrant"], errors["Method"]],
    errors["Outcome"]
).reindex(columns=outcome_order, fill_value=0).reset_index()

summary["Total"] = summary[outcome_order].sum(axis=1)
summary["Execution Failure (%)"] = (
    summary["Execution Failure"] / summary["Total"] * 100
)
executed = summary["Correct"] + summary["Wrong Result"]
summary["Accuracy if Executed (%)"] = (
    summary["Correct"].div(executed.where(executed != 0)) * 100
)

# 3. Ghép bảng đối chiếu GPT-4/GPT-5.4
paper_results = comparison.loc[
    comparison["Quadrant"] != "Overall",
    ["Quadrant", "Method", "GPT-4 (%)", "GPT-5.4 (%)", "Difference (pp)"]
]
summary = summary.merge(paper_results, on=["Quadrant", "Method"],
                        how="left", validate="one_to_one")
assert len(summary) == 8 and summary["GPT-4 (%)"].notna().all()
assert int(summary["Total"].sum()) == 4300

# 4. Xuất bảng evidence cho báo cáo
summary = summary.sort_values(["Quadrant", "Method"])
summary.to_csv(OUT_DIR / "discussion_evidence.csv", index=False, encoding="utf-8-sig")

print("=== PAPER COMPARISON AND ERROR EVIDENCE (50 RUNS) ===")
display(summary[[
    "Quadrant", "Method", "GPT-4 (%)", "GPT-5.4 (%)", "Difference (pp)",
    "Correct", "Wrong Result", "Execution Failure", "Total",
    "Execution Failure (%)", "Accuracy if Executed (%)"
]].round(2))
print("Saved: discussion_evidence.csv")

=== PAPER COMPARISON AND ERROR EVIDENCE (50 RUNS) ===


,Quadrant,Method,GPT-4 (%),GPT-5.4 (%),Difference (pp),Correct,Wrong Result,Execution Failure,Total,Execution Failure (%),Accuracy if Executed (%)
0,HQHS,SPARQL,38.7,44.20,5.50,221,74,205,500,41.00,74.92
1,HQHS,SQL,0.0,49.60,49.60,248,245,7,500,1.40,50.30
2,HQLS,SPARQL,66.9,66.91,0.01,368,106,76,550,13.82,77.64
3,HQLS,SQL,37.4,52.73,15.33,290,210,50,550,9.09,58.00
4,LQHS,SPARQL,35.7,67.20,31.50,336,164,0,500,0.00,67.20
5,LQHS,SQL,0.0,9.80,9.80,49,429,22,500,4.40,10.25
6,LQLS,SPARQL,71.1,80.67,9.57,484,115,1,600,0.17,80.80
7,LQLS,SQL,25.5,66.00,40.50,396,202,2,600,0.33,66.22


Saved: discussion_evidence.csv


## Findings and Limitations (50 runs)

Thực nghiệm tái lập đánh giá GPT-5.4 trên **43 câu hỏi ACME Insurance, 50 lần chạy độc lập**, tổng cộng **4.300 lượt chấm** (2.150 SQL và 2.150 SPARQL). Kết quả AOEA toàn bộ là **45,72% đối với SQL** và **65,53% đối với SPARQL**, chênh lệch **19,81 điểm phần trăm** nghiêng về SPARQL.

**Đối chiếu bảng GPT-4 được lưu trong notebook gốc:** SQL từ **16,70% lên 45,72%** (tăng **29,02 điểm phần trăm**), SPARQL từ **54,20% lên 65,53%** (tăng **11,33 điểm phần trăm**). Chênh lệch SPARQL–SQL thay đổi từ **37,50** xuống **19,81 điểm phần trăm** (giảm **17,69 điểm phần trăm**).

**Theo Quadrant:**

- **LQLS (12 câu):** SQL **66,00%**, SPARQL **80,67%**.
- **HQLS (11 câu):** SQL **52,73%**, SPARQL **66,91%**.
- **LQHS (10 câu):** SQL **9,80%**, SPARQL **67,20%**; mức chênh lệch lớn nhất nghiêng về SPARQL (**57,40 điểm phần trăm**).
- **HQHS (10 câu):** SQL **49,60%**, SPARQL **44,20%**; SQL cao hơn **5,40 điểm phần trăm**.

**Phân tích lỗi:** sử dụng bảng `discussion_evidence.csv` và `execution_failure_taxonomy.csv` được tính lại từ 50 run. Phân biệt riêng truy vấn đúng, truy vấn thực thi được nhưng sai kết quả và truy vấn thực thi thất bại. Không sử dụng lại số lượt lỗi của 5 run cũ. Với stack trace Ontop bị cắt ngắn, không thể khẳng định nguyên nhân gốc chỉ dựa vào tên hàm trong log.

**Giới hạn so sánh:** kết quả GPT-5.4 và bảng GPT-4 trong bài báo có thể khác nhau về mô hình, thiết lập suy luận, thời gian hoặc môi trường thực thi. Vì vậy, mức tăng AOEA chỉ là **đối chiếu quan sát**, không thể quy toàn bộ cho việc thay LLM. Chênh lệch giữa hai nhánh SQL/SPARQL cũng không chứng minh một mình KG gây ra thay đổi hiệu quả, bởi đầu vào prompt và pipeline thực thi khác nhau.

**Hướng tiếp theo:** dựa trên các trường hợp điển hình trong `case_studies.csv` để giải thích lỗi, đối chiếu thí nghiệm Context Ablation và KG Quality Ablation đã hoàn thành, sau đó trình bày đóng góp và giới hạn trong báo cáo.
